In [ ]:
# 2026 Spotify wrapped unwrapped
## FFAM-MDAP Collab
### My streaming history and playlist exploration
#Import
from load_json_files import load_json_files
import json
import ijson
import pandas as pd

import matplotlib.pyplot as plt
plt.rcParams['animation.ffmpeg_path'] = r'D:\Programs\ffmpeg\bin\ffmpeg.exe' # Windows example
import matplotlib.dates as mdates
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Patch, Rectangle

import datetime

from adjustText import adjust_text
import re
from collections import Counter
import os

import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

from pathlib import Path

import urllib.parse

In [ ]:
#plt.rcParams['animation.ffmpeg_path'] = r'D:\Programs\ffmpeg\bin\ffmpeg.exe' # Windows example


In [ ]:
#Helper functions

#Local uri
def parse_spotify_local(uri):
    """
    Decodes the metadata embedded in a spotify:local URI.
    Format: spotify:local:artist:album:track_name:duration
    """
    if pd.isna(uri) or not isinstance(uri, str) or not uri.startswith("spotify:local:"):
        return None
    
    parts = uri.split(':')
    if len(parts) >= 5:
        # The 5th element (index 4) is the track name.
        # We replace '+' with spaces and decode percent-encoding for characters like %E3.
        return urllib.parse.unquote(parts[4].replace('+', ' '))
    return "Unknown Local Track"

#MM:SS to miliseconds conversion
# 1. Define the conversion function
def length_to_ms(length_str):
    try:
        if pd.isna(length_str) or not isinstance(length_str, str):
            return 0
        
        # Split minutes and seconds
        parts = length_str.split(':')
        
        # Handle mm:ss
        if len(parts) == 2:
            minutes = int(parts[0])
            seconds = int(parts[1])
            return (minutes * 60 + seconds) * 1000
        
        # Handle hh:mm:ss (if some longer songs exist)
        elif len(parts) == 3:
            hours = int(parts[0])
            minutes = int(parts[1])
            seconds = int(parts[2])
            return ((hours * 3600) + (minutes * 60) + seconds) * 1000
        
        return 0
    except:
        return 0

#Final vibe, priority to 500k and then 278k
def get_emotion_final(row):
    # Initialize a dictionary for the results
    res = {'emotion_final': 'niche_selection'}
    
    # Priority 1: 500k (Lyrics + Context + Features)
    if pd.notna(row['emotion_500k']):
        res['emotion_final'] = row['emotion_500k']
        # Pull features from 500k columns
        for col in feature_cols:
            res[f'{col}'] = row.get(f'{col}')
            
    # Priority 2: 278k (Features only)
    elif pd.notna(row['emotion_278k']):
        res['emotion_final'] = row['emotion_278k']
        # Pull features from 278k columns
        for col in feature_cols:
            res[f'{col}'] = row.get(f'{col}')
        # # Context remains NaN as 278k doesn't have it
        # for col in context_cols:
        #     res[f'final_{col}'] = None
            
    # Priority 3: Niche (Everything else is NaN)
    else:
        for col in feature_cols:
            res[f'{col}'] = None
            
    return pd.Series(res)

def get_academic_calendar(year):
    """
    Simplified UniMelb logic:
    SWOTVAC: 7 days before June/Nov
    Exams: All of June, All of November
    """
    calendar = {
        'SWOTVAC_1': (datetime.date(year, 5, 25), datetime.date(year, 6, 1)),
        'EXAMS_1':   (datetime.date(year, 6, 1),  datetime.date(year, 6, 30)),
        'SWOTVAC_2': (datetime.date(year, 10, 25), datetime.date(year, 11, 1)),
        'EXAMS_2':   (datetime.date(year, 11, 1),  datetime.date(year, 11, 30)),
    }
    return calendar


def get_daily_signatures(df, column_name):
    # 1. Filter out NaNs and 'niche_selection'
    signal_only = df[df[column_name].notna() & (df[column_name] != 'niche_selection')]
    
    # 2. Group by date and calculate proportions
    daily = signal_only.groupby(['date', column_name])['ms_played'].sum().unstack(fill_value=0)
    #daily_norm = daily.div(daily.sum(axis=1), axis=0) 
    daily_perc = daily.div(daily.sum(axis=1), axis=0) * 100
    
    # 3. Apply a 7-day rolling mean to see the 'tempo' of the change
    return daily_perc.rolling(window=7, min_periods=1).mean()


def get_match_coverage(df, column_name='emotion_final'):
    # Group and count the number of track entries per day
    coverage = df.groupby(['date', column_name]).size().unstack(fill_value=0)
    
    # Identify Niche vs Kaggle-Matched
    niche_count = coverage.get('niche_selection', pd.Series(0, index=coverage.index))
    
    match_cols = [c for c in coverage.columns if c != 'niche_selection']
    kaggle_count = coverage[match_cols].sum(axis=1)
    
    # Apply a 7-day rolling mean to match the "tempo" of the other charts
    df_coverage = pd.DataFrame({
        'Kaggle Match': kaggle_count, 
        'Niche Selection': niche_count
    })
    return df_coverage.rolling(window=7, min_periods=1).mean()





In [ ]:
#load json files
#folder = "C:\Users\User\OneDrive - The University of Melbourne\Unimelb\Semester 3\SCIE90017\SpotifyData\Selected Data"
#folder = "."
root = Path(__file__).parent.resolve() if '__file__' in locals() else Path(os.getcwd())
path = root / "data ama" #"data ama" or "data ken"
print(path)

json_data = load_json_files(path)
print(json_data.keys())

# Load file
#Inferences
inferences_data = json_data['Inferences.json']

inferences = inferences_data["inferences"]  # Extract list
print(f"Total inferences: {len(inferences)}")

#Marquee
marquee = json_data['Marquee.json']

#Library
library = json_data['YourLibrary.json']


In [ ]:
#Kaggle Spotify dataset
#Option 1 278k from moodify with uri
kaggle_278k = root / "kaggle" / "278k_labelled_uri.csv"
kaggle_278k = pd.read_csv(kaggle_278k)
kaggle_278k = kaggle_278k.drop(columns=['Unnamed: 0.1', 'Unnamed: 0', 'spec_rate']) 
#'duration (ms)', 'danceability', 'energy',
#'loudness', 'speechiness', 'acousticness', 'instrumentalness',
#'liveness', 'valence', 'tempo', 'labels', 'uri'

#mood map for kaggle 278k
mood_map = {0: 'sad', 1: 'happy', 2: 'energetic', 3: 'calm'}
kaggle_278k['emotion_mapped'] = kaggle_278k['labels'].map(mood_map).fillna('unknown')
kaggle_278k.rename(columns={'duration (ms)': 'duration_ms'}, inplace=True)

#Option 2 500k from Abracadabra project with isrc
kaggle_500k = root / "kaggle" / "final_milliondataset_BERT_500K_revised.json"
# Updated list to include activity tags
context_cols = [
    'Good for Party', 'Good for Work/Study', 'Good for Relaxation/Meditation',
    'Good for Exercise', 'Good for Running', 'Good for Yoga/Stretching',
    'Good for Driving', 'Good for Social Gatherings', 'Good for Morning Routine'
]
keep_columns = [
    'Length', 'Danceability', 'Energy', 
    'Loudness (db)', 'Speechiness', 'Acousticness', 'Instrumentalness', 
    'Liveness', 'Positiveness', 'Tempo', 'emotion', 'ISRC', 
    'Album', 'song', 'Artist(s)', 'Genre'
    ] + context_cols

#Original just ['Album', 'song', 'Artist(s)', 'Genre', 'ISRC', 'emotion', 'Tempo']
 

# Use lines=True to fix the "trailing garbage" error
# Use chunksize to keep RAM usage low during the load
chunks = pd.read_json(
    kaggle_500k, 
    lines=True, 
    chunksize=10000, 
    encoding='utf-8'
)

# Process chunks and drop the heavy lyrics immediately
df_list = []
for chunk in chunks:
    # We only keep the 4 target columns to save memory
    df_list.append(chunk[keep_columns])

# Final clean dataset for your UniMelb project
kaggle_500k = pd.concat(df_list, ignore_index=True)

print(f"Successfully loaded {len(kaggle_500k)} tracks.")
print(kaggle_500k.head())

In [ ]:
#Preprocess 500k to follow 278k 

# 1. Rename columns
new_columns = [
    'length', 'danceability', 'energy', 
    'loudness', 'speechiness', 'acousticness', 'instrumentalness',
    'liveness', 'valence', 'tempo', 'emotion', 'isrc',
    'album', 'song', 'artists', 'genre'
    ] + context_cols

kaggle_500k.columns = new_columns   

# 2. Create the unification dictionary
unify_map = {
    'joy': 'joy',
    'love': 'love',
    'sadness': 'sadness', 
    'anger': 'anger', #from here below is energetic if we want to follow same emotion as 278k
    'angry': 'anger',
    'fear': 'fear',
    'surprise': 'surprise',
    'confusion': 'confusion',
}

# 3. Apply the mapping to the 500k dataset
# We lower() first to catch 'Love' and 'love' in one go
kaggle_500k['emotion_mapped'] = kaggle_500k['emotion'].str.lower().map(unify_map)

# 4. Change length to duration (ms)
# Replace 'kaggle_500k' with your actual variable name
kaggle_500k['duration_ms'] = kaggle_500k['length'].apply(length_to_ms)

# 4. (Optional) Remove the old string column to save memory
# kaggle_500k.drop(columns=['length'], inplace=True)

print(kaggle_500k[['song', 'duration_ms']].head())




In [ ]:
#Standardise audio feature numerical values to between 0 and 1
# List of audio features common to both
audio_features = ['danceability', 'energy', 'speechiness', 'acousticness', 
                  'instrumentalness', 'liveness', 'valence', 'tempo']

# 500k is likely 0-100, so we divide by 100. 
# (Tempo is usually 40-200+, so we don't divide that one)
for feature in audio_features:
    if feature != 'tempo':
        kaggle_500k[feature] = kaggle_500k[feature] / 100.0

In [ ]:
print(kaggle_278k.dtypes)
print(kaggle_278k.columns)
print(kaggle_278k.head())

In [ ]:
print(kaggle_500k.dtypes)
print(kaggle_500k.columns)
print(kaggle_500k.head())
print(sorted(kaggle_500k["emotion"].unique()))

In [ ]:
values_to_drop = ['pink', 'thirst', 'True', 'interest','confusion']
print(len(kaggle_500k[kaggle_500k['emotion'].isin(values_to_drop)]))
print(kaggle_500k[kaggle_500k['emotion'].isin(values_to_drop)]) #only 21, drop them

kaggle_500k = kaggle_500k[~kaggle_500k['emotion'].isin(values_to_drop)]

In [ ]:
# Find duplicates based on 'Column1' and 'Column2'
is_duplicate = kaggle_500k.duplicated(subset=['song', 'artists'])

# View the actual duplicate rows
duplicate_rows = kaggle_500k[is_duplicate]

# Check for rows where EVERY column is the same
true_duplicates_count = kaggle_500k.duplicated().sum()

print(f"Total Rows: {len(kaggle_500k)}")
print(f"Rows with same Song/Artist: {len(duplicate_rows)}")
print(f"Rows that are 100% identical across all columns: {true_duplicates_count}")

#since they are the same we can drop duplicates

kaggle_500k = kaggle_500k.drop_duplicates()
print(f"Total Rows after remove duplicates: {len(kaggle_500k)}")

In [ ]:
#Playlist data
playlist_data = []
# explore playlist data
files = [fname for fname in json_data.keys() if 'Playlist' in fname]
for f in files:
    print(f)
    playlist_data.append(json_data[f])

print(playlist_data[0]['playlists'][0].keys())
playlistdf = pd.json_normalize(playlist_data[0]['playlists'])

#tracksdf = pd.json_normalize(playlist_data[0]['playlists'], record_path='items', meta=['name', 'lastModifiedDate', 'description'])

# # Combine all playlists from all files
# 1. Generate the base items DataFrame
all_items = []
all_items.append(pd.json_normalize(
    playlist_data[0]['playlists'],
    record_path='items',
    meta=['name', 'lastModifiedDate', 'description'],
    errors='ignore'
))
items_df = pd.concat(all_items, ignore_index=True)

# 2. Identify available local track columns
# json_normalize creates columns based on the JSON keys found.
local_col = 'localTrack.uri' if 'localTrack.uri' in items_df.columns else None
name_col = 'track.trackName'

# 3. Apply the fallback logic
if local_col:
    # Extract names from local URIs for rows where they exist
    items_df['local_name_extracted'] = items_df[local_col].apply(parse_spotify_local)
    
    # Use fillna to prioritize the official trackName, falling back to the local name
    if 'track.trackName' not in items_df.columns:
        items_df['track.trackName'] = None
    items_df['track.trackName'] = items_df[name_col].fillna(items_df['local_name_extracted'])

# 4. Cleanup and Verification
print(f"Final Column List: {items_df.columns.tolist()}")
print('--- Sample of Unified Track Names ---')
print(items_df[['name', 'track.trackName']].sample(min(10, len(items_df))))

In [ ]:
#Streaming data
# get all streaming history

streaming_data = []
# explore streaming data
files = [fname for fname in json_data.keys() if 'Streaming' in fname]
for f in files:
    print(f)
    streaming_data.append(json_data[f])

streaminghistorydf = pd.concat(streaming_data, ignore_index=True)

print(streaminghistorydf.columns)
print(streaminghistorydf.dtypes)
print(streaminghistorydf.count())

#streaminghistorydf = pd.DataFrame(json_data[recentfile])
streaminghistorydf['ts'] = pd.to_datetime(streaminghistorydf['ts'])
streaminghistorydf['date'] = streaminghistorydf['ts'].dt.date
streaminghistorydf['year'] = streaminghistorydf['ts'].dt.year
streaminghistorydf['month'] = streaminghistorydf['ts'].dt.month
streaminghistorydf['day'] = streaminghistorydf['ts'].dt.to_period('D')
streaminghistorydf['hour'] = streaminghistorydf['ts'].dt.hour
streaminghistorydf['minutes'] = streaminghistorydf['ts'].dt.minute

#streaminghistorydf['minutes'] = streaminghistorydf['ms_played'] / 60000
print(streaminghistorydf.columns)

# Create a mapping from track URI to playlist names (handles multiple playlists per track)
if 'track.trackUri' not in items_df.columns:
    items_df['track.trackUri'] = items_df['localTrack.uri']
track_to_playlists = items_df.groupby('track.trackUri')['name'].apply(list)

# Map track URIs to playlist names
streaminghistorydf['playlist'] = streaminghistorydf['spotify_track_uri'].map(track_to_playlists)

# Add flag for whether track is from a playlist
streaminghistorydf['is_from_playlist'] = streaminghistorydf['playlist'].notna()




In [ ]:
# Add emotion features from kaggle dataset to streaming history
# First Pass: Match by URI to pull in the 278k data.
# Second Pass: Match by Names to pull in the 500k data.
# --- 1. PREP THE DATASETS ---

# Standardize 500k for Name-matching
df_500k_ref = kaggle_500k[['song', 'artists', 'isrc', 'emotion_mapped'] + context_cols].copy()
df_500k_ref['track_clean'] = df_500k_ref['song'].str.lower().str.strip()
df_500k_ref['artist_clean'] = df_500k_ref['artists'].str.lower().str.strip()
# Drop duplicates to avoid row-ballooning
df_500k_ref = df_500k_ref.drop_duplicates(subset=['track_clean', 'artist_clean'])

# --- 2. THE DOUBLE JOIN ---
# Join A: Use the URI to get Audio labels (278k)
# History URI column: 'spotify_track_uri'
history_enriched = pd.merge(
    streaminghistorydf, 
    kaggle_278k[['uri', 'emotion_mapped']].rename(columns={'emotion_mapped': 'emotion_278k'}),
    left_on='spotify_track_uri', 
    right_on='uri', 
    how='left'
)

# Join B: Use the Names to get Lyric labels (500k)
# History Name columns: 'master_metadata_track_name' and 'master_metadata_album_artist_name'
history_enriched['track_clean'] = history_enriched['master_metadata_track_name'].str.lower().str.strip()
history_enriched['artist_clean'] = history_enriched['master_metadata_album_artist_name'].str.lower().str.strip()

history_enriched = pd.merge(
    history_enriched,
    df_500k_ref[['track_clean', 'artist_clean', 'isrc', 'emotion_mapped'] + context_cols].rename(columns={'emotion_mapped': 'emotion_500k'}),
    on=['track_clean', 'artist_clean'],
    how='left'
)

In [ ]:
# Create a mask where at least one mood column is not NaN
emotion_found_mask = history_enriched['emotion_278k'].notna() | history_enriched['emotion_500k'].notna()

# Filter the dataframe
history_with_moods = history_enriched[emotion_found_mask]

# View the result
print(f"Total history rows: {len(history_enriched)}")
print(f"Rows with at least one mood: {len(history_with_moods)}")
print(f"Success Rate: {(len(history_with_moods) / len(history_enriched)) * 100:.2f}%")

# Songs with BOTH 278k and 500k moods (Highest confidence)
both = history_enriched[history_enriched['emotion_278k'].notna() & history_enriched['emotion_500k'].notna()]

# Songs with ONLY 278k mood
emo_278k_only = history_enriched[history_enriched['emotion_278k'].notna() & history_enriched['emotion_500k'].isna()]

# Songs with ONLY 500k mood
emo_500k_only = history_enriched[history_enriched['emotion_278k'].isna() & history_enriched['emotion_500k'].notna()]

print(f"Both (each can be different): {len(both)}")
print(f"278k Only: {len(emo_278k_only)}")
print(f"500k Only: {len(emo_500k_only)}")

history_with_moods.head()

In [ ]:
#Comparing the Signatures: The "Agreement" Matrix
# Filter for the "High Confidence" rows where both have labels
comparison_df = history_enriched.dropna(subset=['emotion_278k', 'emotion_500k'])

# Create a cross-tabulation (The Signature Overlap)
overlap_matrix = pd.crosstab(comparison_df['emotion_278k'], comparison_df['emotion_500k'], normalize='index')

plt.figure(figsize=(10, 8))
sns.heatmap(overlap_matrix, annot=True, cmap='YlGnBu')
plt.title('How the 278k Audio Labels map to 500k Lyric Emotions')
plt.show()

In [ ]:
#Use 500k first and then 278k for emotions since 500k seems more meaningful than 278k
# The audio features common to both datasets
feature_cols = [
    'danceability', 'energy', 'valence', 'tempo', 
    'speechiness', 'acousticness', 'instrumentalness', 'liveness'
]

# Apply the function to create the new columns
vibe_features_df = history_enriched.apply(get_emotion_final, axis=1)

# Join the results back to your main history
history_enriched = pd.concat([history_enriched, vibe_features_df], axis=1)

In [ ]:
print(history_enriched.dtypes)
print(history_enriched.columns)

In [ ]:
history_enriched.to_csv("history_enriched_v4.csv")

In [ ]:
# Generate the two separate signature timelines
sig_500k = get_daily_signatures(history_enriched, 'emotion_500k')
sig_278k = get_daily_signatures(history_enriched, 'emotion_278k')

In [ ]:
#Daily Emotions
#Duration-Weighted Mode

# 1. Group by Date AND Emotion, then sum the Duration
# We use 'emotion_final' (the one where you prioritized 500k's detailed labels)
daily_emo_totals = (
    history_enriched.groupby(['date', 'emotion_final'])['ms_played']
    .sum()
    .reset_index()
)

# 2. SEPARATE the "Signal" (Joy, Sad, etc.) from the "Noise" (niche_selection)
signal_data = daily_emo_totals[daily_emo_totals['emotion_final'] != 'niche_selection']
niche_only_data = daily_emo_totals[daily_emo_totals['emotion_final'] == 'niche_selection']

# 3. Find the Winner for days that HAVE a signal
# This ignores 'niche_selection' even if it had more duration
daily_vibe_signal = (
    signal_data.sort_values('ms_played', ascending=False)
    .drop_duplicates('date')
    .rename(columns={'emotion_final': 'daily_dominant_mood', 'ms_played': 'mood_duration_ms'})
)

# 4. Find the dates that were NOT in the signal group
all_dates = daily_emo_totals['date'].unique()
signal_dates = daily_vibe_signal['date'].unique()
missing_dates = set(all_dates) - set(signal_dates)

# 5. Create a "Niche" dataframe for those missing dates
niche_backfill = niche_only_data[niche_only_data['date'].isin(missing_dates)].copy()

# Rename columns to match the signal dataframe for a clean concat
niche_backfill = niche_backfill.rename(columns={
    'emotion_final': 'daily_dominant_mood',
    'ms_played': 'mood_duration_ms'
})

# 6. COMBINE: Signal winners + Niche fallbacks
daily_vibe_final = pd.concat([
    daily_vibe_signal[['date', 'daily_dominant_mood', 'mood_duration_ms']], 
    niche_backfill[['date', 'daily_dominant_mood', 'mood_duration_ms']]
    ])

# Sort for a chronological timeline
daily_vibe_final = daily_vibe_final.sort_values('date').reset_index(drop=True)

print(f"Total days analyzed: {len(daily_vibe_final)}")
print(daily_vibe_final['daily_dominant_mood'].value_counts())

In [ ]:
daily_vibe_final.to_csv("daily_vibe_final.csv")

In [ ]:
# Master list for shading (prevents disappearance at year-end)
    #a Master Calendar for all years
all_years_cal = []
for y in range(2015, 2027):
    # Get the dictionary for each year and add to a list
    y_cal = get_academic_calendar(y)
    all_years_cal.extend(y_cal.values())

def create_dashboard_animation(sig_500k, sig_278k, output_name="final_unimelb_dashboard.mp4"):
    common_dates = sig_500k.index.intersection(sig_278k.index)
    data500 = sig_500k.loc[common_dates]
    data278 = sig_278k.loc[common_dates]
    
    pos_tags = ['joy', 'love', 'happy', 'calm', 'energetic']
    neg_tags = ['anger', 'sadness', 'fear', 'sad']
    
    # Pre-calculate Average Sentiment
    all_pos = (data500[data500.columns.intersection(pos_tags)].sum(axis=1) + 
               data278[data278.columns.intersection(pos_tags)].sum(axis=1)) / 2
    all_neg = (data500[data500.columns.intersection(neg_tags)].sum(axis=1) + 
               data278[data278.columns.intersection(neg_tags)].sum(axis=1)) / 2

    fig = plt.figure(figsize=(20, 16))
    gs = fig.add_gridspec(3, 2, height_ratios=[1, 1, 0.8])
    ax1, ax2 = fig.add_subplot(gs[0, 0], projection='polar'), fig.add_subplot(gs[0, 1], projection='polar')
    ax3, ax4, ax5 = fig.add_subplot(gs[1, 0]), fig.add_subplot(gs[1, 1]), fig.add_subplot(gs[2, :])
    plt.subplots_adjust(hspace=0.4, wspace=0.2)

    color_map = {
        'love': '#FF69B4', 'joy': '#FFD700', 'fear': '#228B22', 'surprise': '#00BFFF', 
        'sadness': '#0000FF', 'anger': '#FF0000', 'energetic': '#FFA500', 
        'sad': '#4169E1', 'calm': '#B0C4DE', 'happy': '#87CEEB',
        'Positive': '#32CD32', 'Negative': '#DC143C',
        'SWOTVAC_bg': '#FFF9C4', 'EXAM_bg': '#FFEBEE'
    }

    def update(frame):
        current_dt = common_dates[frame]
        end_view = pd.to_datetime(current_dt).date()
        start_view = end_view - datetime.timedelta(days=45)
        
        # Mask the data to only include the 45-day calendar window
        view_mask = (common_dates >= start_view) & (common_dates <= end_view)
        view_dates = common_dates[view_mask]

        # --- Subplots Cleaning & Shading ---
        for ax in [ax1, ax2, ax3, ax4, ax5]: ax.clear()
        for ax in [ax3, ax4, ax5]:
            ax.set_xlim(start_view, end_view)
            for (s, e) in all_years_cal:
                if s <= end_view and e >= start_view:
                    color = color_map['EXAM_bg'] if s.month in [6, 11] else color_map['SWOTVAC_bg']
                    ax.axvspan(s, e, color=color, alpha=0.7, zorder=0)

        # --- Row 1: Radar (Current State Only) ---
        for ax, data, title in zip([ax1, ax2], [data500, data278], ["LYRICS (%)", "AUDIO (%)"]):
            row = data.iloc[frame]
            angles = np.linspace(0, 2*np.pi, len(row), endpoint=False)
            ax.bar(angles, row.values, width=0.7, color=[color_map.get(l, 'gray') for l in row.index], alpha=0.8)
            ax.set_theta_offset(np.pi/2); ax.set_theta_direction(-1); ax.set_ylim(0, 100)
            ax.set_xticks(angles); ax.set_xticklabels([f"{l}\n{v:.0f}%" for l, v in zip(row.index, row.values)], size=8, weight='bold')
            ax.set_title(f"{title}\n{current_dt}", weight='bold', pad=25)

        # --- Row 2 & 3: Temporal Trends (Using view_dates) ---
        for col in data500.columns:
            ax3.plot(view_dates, data500.loc[view_dates, col], color=color_map.get(col), lw=2, label=col)
        for col in data278.columns:
            ax4.plot(view_dates, data278.loc[view_dates, col], color=color_map.get(col), lw=2, label=col)
        
        ax5.bar(view_dates, all_pos.loc[view_dates], color=color_map['Positive'], alpha=0.7)
        ax5.bar(view_dates, -all_neg.loc[view_dates], color=color_map['Negative'], alpha=0.7)
        ax5.axhline(0, color='black', lw=1)

        # Final Formatting
        ax3.set_title("LYRICS TRENDS", weight='bold'); ax4.set_title("AUDIO TRENDS", weight='bold')
        ax5.set_title("SENTIMENT TUG-OF-WAR", weight='bold'); ax5.set_ylim(-100, 100)
        for ax in [ax3, ax4, ax5]:
            ax.spines[['top', 'right']].set_visible(False)
            ax.tick_params(axis='x', rotation=25)
            ax.set_ylim(0, 100) if ax != ax5 else None

    ani = FuncAnimation(fig, update, frames=len(common_dates), interval=100)
    ani.save(output_name, writer='ffmpeg', fps=10)
    plt.close()

In [ ]:
# Master list for shading (prevents disappearance at year-end)
    #a Master Calendar for all years
all_years_cal = []
for y in range(2015, 2027):
    # Get the dictionary for each year and add to a list
    y_cal = get_academic_calendar(y)
    all_years_cal.extend(y_cal.values())

#Kaggle vs Niche pre-calculation
coverage_data = get_match_coverage(history_enriched, 'emotion_final') # or 500k or 278k

def create_dashboard_animation(sig_500k, sig_278k, output_name="final_unimelb_dashboard.mp4"):
    common_dates = sig_500k.index.intersection(sig_278k.index)
    data500 = sig_500k.loc[common_dates]
    data278 = sig_278k.loc[common_dates]
    
    pos_tags = ['joy', 'love', 'happy', 'calm', 'energetic']
    neg_tags = ['anger', 'sadness', 'fear', 'sad']
    
    # Pre-calculate Average Sentiment
    all_pos = (data500[data500.columns.intersection(pos_tags)].sum(axis=1) + 
               data278[data278.columns.intersection(pos_tags)].sum(axis=1)) / 2
    all_neg = (data500[data500.columns.intersection(neg_tags)].sum(axis=1) + 
               data278[data278.columns.intersection(neg_tags)].sum(axis=1)) / 2

    # fig = plt.figure(figsize=(20, 16))
    # gs = fig.add_gridspec(3, 2, height_ratios=[1, 1, 0.8])
    fig = plt.figure(figsize=(22, 18))
    gs = fig.add_gridspec(3, 2, height_ratios=[1, 1, 1])
    # ax1, ax2 = fig.add_subplot(gs[0, 0], projection='polar'), fig.add_subplot(gs[0, 1], projection='polar')
    # ax3, ax4, ax5 = fig.add_subplot(gs[1, 0]), fig.add_subplot(gs[1, 1]), fig.add_subplot(gs[2, :])
    # plt.subplots_adjust(hspace=0.4, wspace=0.2)

    ax1, ax2 = fig.add_subplot(gs[0, 0], projection='polar'), fig.add_subplot(gs[0, 1], projection='polar')
    ax3, ax4 = fig.add_subplot(gs[1, 0]), fig.add_subplot(gs[1, 1])
    ax5 = fig.add_subplot(gs[2, 0]) # Sentiment Mirror
    ax6 = fig.add_subplot(gs[2, 1]) # NEW: Data Match Volume
    plt.subplots_adjust(hspace=0.4, wspace=0.2)

    color_map = {
        'love': '#FF69B4', 'joy': '#FFD700', 'fear': '#228B22', 'surprise': '#00BFFF', 
        'sadness': '#0000FF', 'anger': '#FF0000', 'energetic': '#FFA500', 
        'sad': '#4169E1', 'calm': '#B0C4DE', 'happy': '#87CEEB',
        'Positive': '#32CD32', 'Negative': '#DC143C',
        'SWOTVAC_bg': '#FFF9C4', 'EXAM_bg': '#FFEBEE'
    }

    def update(frame):
        current_dt = common_dates[frame]
        end_view = pd.to_datetime(current_dt).date()
        start_view = end_view - datetime.timedelta(days=45)
        
        # Mask the data to only include the 45-day calendar window
        view_mask = (common_dates >= start_view) & (common_dates <= end_view)
        view_dates = common_dates[view_mask]

        # --- Subplots Cleaning & Shading ---
        for ax in [ax1, ax2, ax3, ax4, ax5, ax6]: ax.clear()
        for ax in [ax3, ax4, ax5, ax6]:
            ax.set_xlim(start_view, end_view)
            for (s, e) in all_years_cal:
                if s <= end_view and e >= start_view:
                    color = color_map['EXAM_bg'] if s.month in [6, 11] else color_map['SWOTVAC_bg']
                    ax.axvspan(s, e, color=color, alpha=0.7, zorder=0)

        # --- Row 1: Radar (Current State Only) ---
        for ax, data, title in zip([ax1, ax2], [data500, data278], ["LYRICS (%)", "AUDIO (%)"]):
            row = data.iloc[frame]
            angles = np.linspace(0, 2*np.pi, len(row), endpoint=False)
            ax.bar(angles, row.values, width=0.7, color=[color_map.get(l, 'gray') for l in row.index], alpha=0.8)
            ax.set_theta_offset(np.pi/2); ax.set_theta_direction(-1); ax.set_ylim(0, 100)
            ax.set_xticks(angles); ax.set_xticklabels([f"{l}\n{v:.0f}%" for l, v in zip(row.index, row.values)], size=8, weight='bold')
            ax.set_title(f"{title}\n{current_dt}", weight='bold', pad=25)

        # --- PANEL 3 & 4: TREND LINES (Fixed 0-100%) ---
        for col in data500.columns:
            ax3.plot(view_dates, data500.loc[view_dates, col], color=color_map.get(col), lw=2)
        for col in data278.columns:
            ax4.plot(view_dates, data278.loc[view_dates, col], color=color_map.get(col), lw=2)
        ax3.set_ylim(0, 100); ax4.set_ylim(0, 100)

        # --- PANEL 5: SENTIMENT TUG-OF-WAR (Fixed -100 to 100%) ---
        ax5.bar(view_dates, all_pos.loc[view_dates], color=color_map['Positive'], alpha=0.7)
        ax5.bar(view_dates, -all_neg.loc[view_dates], color=color_map['Negative'], alpha=0.7)
        ax5.set_ylim(-100, 100)

        # --- PANEL 6: MATCH COVERAGE (Fixed 0-150 Tracks) ---
        v_cov = coverage_data.loc[view_dates]
        ax6.stackplot(view_dates, v_cov['Kaggle Match'], v_cov['Niche Selection'], 
                    labels=['Kaggle Match', 'Niche Selection'],
                    colors=['#4682B4', '#D3D3D3'], alpha=0.8)
        
        ax6.set_ylim(0, 100) 
        ax6.set_title("GLOBAL DATA VOLUME (Track Count, Stacked Chart)", weight='bold')
        ax6.legend(loc='upper left', fontsize=8, frameon=False)

        # Styling cleanup for all timelines
        for ax in [ax3, ax4, ax5, ax6]:
            ax.spines[['top', 'right']].set_visible(False)
            ax.tick_params(axis='x', rotation=25)

    ani = FuncAnimation(fig, update, frames=len(common_dates), interval=100)
    ani.save(output_name, writer='ffmpeg', fps=10)
    plt.close()

In [ ]:
#create_side_by_side_animation(sig_500k, sig_278k)
#create_mirrored_animation(sig_500k, sig_278k)
create_dashboard_animation(sig_500k, sig_278k)

In [ ]:
#Keep continuity of variable names on previous graphs
streaminghistorydf = history_enriched

In [ ]:
#First graph Daily Minutes Played: Playlist vs Non-Playlist Songs
# Aggregate by day and playlist status
monthly_streams = streaminghistorydf.groupby(['day', 'is_from_playlist']).agg({
    'ms_played': lambda x: round(x.sum() / 60000),  # Convert to minutes
}).reset_index()
monthly_streams.rename(columns={'ms_played': 'minutes_played'}, inplace=True)

# Pivot to separate columns for playlist vs non-playlist
monthly_streams_pivot = monthly_streams.pivot(index='day', columns='is_from_playlist', values='minutes_played').fillna(0)
print(monthly_streams_pivot)



monthly_streams_pivot.columns = ['non_playlist', 'playlist']
monthly_streams_pivot = monthly_streams_pivot.reset_index()

# show a graph of daily minutes played
plt.figure(figsize=(12, 6))

# Convert period to timestamp for plotting
monthly_streams_pivot['date'] = monthly_streams_pivot['day'].apply(lambda x: x.to_timestamp())

# filter to last 12 months
one_year_ago = pd.Timestamp.now() - pd.DateOffset(months=12)
monthly_streams_pivot = monthly_streams_pivot[monthly_streams_pivot['date'] >= one_year_ago]

# Calculate total minutes and find peak day for each month
monthly_streams_pivot['total_minutes'] = monthly_streams_pivot['playlist'] + monthly_streams_pivot['non_playlist']
monthly_streams_pivot['year_month'] = monthly_streams_pivot['date'].dt.to_period('M')
max_days_idx = monthly_streams_pivot.groupby('year_month')['total_minutes'].idxmax()
max_days = monthly_streams_pivot.loc[max_days_idx]

# Plot stacked area chart
plt.fill_between(monthly_streams_pivot['date'], 0, monthly_streams_pivot['playlist'], 
                 alpha=0.6, color='steelblue', label='From Playlists')
plt.fill_between(monthly_streams_pivot['date'], monthly_streams_pivot['playlist'], 
                 monthly_streams_pivot['playlist'] + monthly_streams_pivot['non_playlist'], 
                 alpha=0.6, color='coral', label='Not in Playlists')

# Add line plots on top
plt.plot(monthly_streams_pivot['date'], monthly_streams_pivot['playlist'], 
         color='darkblue', linewidth=1, alpha=0.8)
plt.plot(monthly_streams_pivot['date'], monthly_streams_pivot['playlist'] + monthly_streams_pivot['non_playlist'], 
         color='darkred', linewidth=1, alpha=0.8)

# Highlight peak days
plt.scatter(max_days['date'], max_days['total_minutes'], color='purple', s=50, zorder=5, label='Peak day each month')

# Add annotations for peak days
for idx, row in max_days.iterrows():
    playlist_pct = (row['playlist'] / row['total_minutes'] * 100) if row['total_minutes'] > 0 else 0
    non_playlist_pct = (row['non_playlist'] / row['total_minutes'] * 100) if row['total_minutes'] > 0 else 0
    annotation_text = f"{row['date'].strftime('%a %d %b')}\nPlaylist: {int(row['playlist'])}m ({playlist_pct:.0f}%)\nNon-playlist: {int(row['non_playlist'])}m ({non_playlist_pct:.0f}%)"
    plt.annotate(annotation_text, 
                 xy=(row['date'], row['total_minutes']),
                 xytext=(0, 10),  # 10 points above
                 textcoords='offset points',
                 ha='center',
                 fontsize=7,
                 color='purple',
                 bbox=dict(boxstyle='round,pad=0.4', facecolor='white', edgecolor='purple', alpha=0.8))

# Format x-axis to show monthly ticks
ax = plt.gca()
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%b'))
plt.xticks(rotation=45)

plt.title('Daily Minutes Played: Playlist vs Non-Playlist Songs')
plt.xlabel('Month')
plt.ylabel('Minutes Played')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('playlistvsother.png')
plt.show()


In [ ]:
#Explode playlist
df_exploded = streaminghistorydf.explode('playlist')
df_exploded['playlist'] = df_exploded['playlist'].fillna(value = "Non_Playlist")  # Drop empty/NaN

In [ ]:
#Second graph 7 day rolling average
# Daily minutes with non-playlist
daily_playlist = (
    df_exploded
    .groupby(['day', 'playlist'])['ms_played']
    .sum().unstack(fill_value=0) / 60000
)

playlist_totals = daily_playlist.sum()
top5_playlists = playlist_totals.drop('Non_Playlist', errors='ignore').nlargest(5).index

# Top 5 Playlists and Others (exclude Non_Playlist baseline)
daily_top5_others = daily_playlist[top5_playlists].copy()
daily_top5_others['Other_Playlists'] = daily_playlist.drop(columns=['Non_Playlist'] + top5_playlists.tolist(), errors='ignore').sum(axis=1)
daily_top5_others['Non_Playlist'] = daily_playlist['Non_Playlist']

# Top 5 + "Other_Playlists" + "Non_playlist" (optional)
playlist_cols = top5_playlists.tolist() #+ ["Other_Playlists"] #+ ['Non_Playlist'] 
daily_top6 = daily_top5_others[playlist_cols]#.tail(365)  # .tail(365) for Last year

# Stacked AREA chart
daily_smooth = daily_top6.rolling(window=7, center=True).mean().iloc[-100:]  # 7-day rolling
daily_smooth.plot(kind='area', figsize=(14, 8), linewidth=1.5, alpha=0.8)
plt.title('Daily Listening (7-Day Rolling Average): Top 5')
plt.ylabel('Minutes')
plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('playlist_area_evolution_rolling.png', dpi=300)
plt.show()


In [ ]:
#Third graph Daily averages by playlist

# Your grouped data (daily totals per hour-playlist)
daily_hourly_playlist = df_exploded[df_exploded['playlist'].notna()].groupby(['day', 'hour', 'playlist'])['minutes'].sum().reset_index()

# Top 3 playlists
playlist_totals = daily_hourly_playlist.groupby('playlist')['minutes'].sum()
top3_playlists = playlist_totals.nlargest(3).index.tolist()
print(f" Top 3: {top3_playlists}")

# **DAILY AVERAGES** across days per hour-playlist
hourly_daily_avg = (
    daily_hourly_playlist
    .groupby(['hour', 'playlist'])['minutes']
    .mean()  # Average across ALL days
    .unstack(fill_value=0)
)

# Add "Other Playlists" (everything minus top 3)
hourly_daily_avg['Other Playlists'] = (
    hourly_daily_avg.sum(axis=1) - 
    hourly_daily_avg[top3_playlists].sum(axis=1)
)
playlists_to_show = top3_playlists #+ ['Other Playlists']

print("\nDaily avg min/hour:")
print(hourly_daily_avg[playlists_to_show].round(1))

# 3 POLAR WHEELS
fig = plt.figure(figsize=(18, 6))

for i, playlist_name in enumerate(playlists_to_show):
    ax = fig.add_subplot(1, 3, i+1, projection='polar')
    
    # Daily avg data
    data = hourly_daily_avg[playlist_name].reindex(range(24), fill_value=0)
    
    # Polar setup
    angles = np.deg2rad(np.arange(0, 360, 15))
    widths = np.deg2rad(np.full(24, 15))

    # 1. Define Day and Night hours (example: 6 AM to 6 PM)
    # In polar coordinates, angles are calculated as radians:
    # angle = (hour / 24) * 2 * np.pi
    day_start, day_end = 6, 18 

    # 2. Create the "Day" background (Yellow/Gold)
    ax.fill_between(
        np.linspace(np.deg2rad(day_start * 15), np.deg2rad(day_end * 15), 100),
        0, ax.get_ylim()[1] if ax.get_ylim()[1] > 0 else 1, 
        color='gold', alpha=0.1, zorder=0, label='Daylight'
    )

    # 3. Create the "Night" background (Midnight Blue)
    # This handles the wrap-around from 18h to 6h
    night_angles = np.linspace(np.deg2rad(day_end * 15), np.deg2rad((day_start + 24) * 15), 100)
    ax.fill_between(
        night_angles,
        0, ax.get_ylim()[1] if ax.get_ylim()[1] > 0 else 1, 
        color='midnightblue', alpha=0.1, zorder=0, label='Night'
    )

        
    # Colors per playlist
    colors = ['#2E8B57', '#FF6347', '#4682B4', '#808080']  # SeaGreen, Tomato, SteelBlue, Gray
    bar_color = colors[i]
    
    bars = ax.bar(angles, data.values, widths, alpha=0.85, 
                  color=bar_color, edgecolor='white', linewidth=1.2)
    
    # Peak highlight
    peak_idx = data.idxmax()
    peak_val = data.max()
    if peak_val > 0:
        ax.bar(angles[peak_idx], peak_val, widths[peak_idx], 
               color='gold', edgecolor='darkorange', linewidth=2.5, zorder=10,
               alpha=1, hatch='//')
        
        # # Peak text
        # mult = peak_val / data.mean() if data.mean() > 0 else 0
        # ax.text(angles[peak_idx], peak_val * 1.05, f'{peak_val:.0f}\n({mult:.1f}x)',
        #         ha='center', va='bottom', weight='bold', fontsize=10,
        #         bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.9))
    
    # Styling
    ax.set_xticks(angles)
    ax.set_xticklabels([f'{h:02d}' for h in range(24)], fontsize=12, weight='bold')
    ax.set_ylim(0, max(data.max() * 1.25, 1))
    ax.grid(True, alpha=0.4, color='gray')
    
    # Stats title
    avg_daily = data.mean()
    total_h = playlist_totals.get(playlist_name, 
                                 playlist_totals[~playlist_totals.index.isin(top3_playlists)].sum()) / 60
    ax.set_title(f'{playlist_name.replace("_", " ").title()}\n'
                 f'{avg_daily:.1f}min/day\n'
                 f'{total_h:.0f}h total',
                 fontsize=11, weight='bold', pad=15, color=bar_color)

plt.suptitle('Daily Listening Patterns (Avg across All Days)\n'
             'Top 3 Playlists + All Others', fontsize=20, weight='bold', y=1.02)
plt.tight_layout()
plt.savefig('4_playlist_wheels_final.png', dpi=400, bbox_inches='tight', facecolor='white')
plt.show()


In [ ]:
# 3 POLAR WHEELS
fig = plt.figure(figsize=(18, 6))

for i, playlist_name in enumerate(playlists_to_show):
    ax = fig.add_subplot(1, 3, i+1, projection='polar')
    
    # 1. CLOCK ORIENTATION: 0 at bottom, 12 at top, moving clockwise
    ax.set_theta_zero_location('S') 
    ax.set_theta_direction(-1)
    
    # 2. Data Preparation
    data = hourly_daily_avg[playlist_name].reindex(range(24), fill_value=0)
    
    # Calculate ymax first so backgrounds and bars scale together
    ymax = max(data.max() * 1.3, 1) 
    
    # 3. DAY/NIGHT BACKGROUNDS (Matching the top/bottom split)
    # Day (Top Half: 6 to 18): Center at 12:00 (pi radians), Width 12 hours (pi radians)
    ax.bar(np.pi, ymax, width=np.pi, color='gold', alpha=0.5, zorder=0)
    
    # Night (Bottom Half: 18 to 6): Center at 00:00 (0 radians), Width 12 hours (pi radians)
    ax.bar(0, ymax, width=np.pi, color='midnightblue', alpha=0.5, zorder=0)

    ## 3. FIXING RADIAL LABELS (The 200, 300, 400 fix)
    # Find the hour with the LEAST listening to place the labels there
    min_hour = data.idxmin()
    # Convert hour to degrees: (hour/24 * 360)
    label_angle = (min_hour / 24) * 360
    
    # Move the 200, 300, 400 labels to that specific angle
    ax.set_rlabel_position(label_angle)
    
    # 4. PLOTTING DATA
    angles = np.deg2rad(np.arange(0, 360, 15))
    widths = np.deg2rad(np.full(24, 15))

    colors = ['#2E8B57', '#FF6347', '#4682B4', '#808080'] 
    bar_color = colors[i]
    
    # Main listening bars
    ax.bar(angles, data.values, widths, alpha=0.85, 
           color=bar_color, edgecolor='white', linewidth=1.2, zorder=5)
    
    # 5. PEAK HIGHLIGHT
    peak_idx = data.idxmax()
    peak_val = data.max()
    if peak_val > 0:
        ax.bar(angles[peak_idx], peak_val, widths[peak_idx], 
               color='gold', edgecolor='darkorange', linewidth=2.5, zorder=10,
               alpha=1, hatch='//')
    
    # 6. STYLING
    ax.set_xticks(angles)
    ax.set_xticklabels([f'{h}' for h in range(24)], fontsize=11, weight='bold')
    ax.set_ylim(0, ymax)
    ax.grid(True, alpha=0.3, color='gray')
    
    # Title & Stats
    avg_daily = data.mean()
    total_h = playlist_totals.get(playlist_name, 0) / 60
    ax.set_title(f'{playlist_name.replace("_", " ").title()}\n'
                 f'{avg_daily:.1f}min/day | {total_h:.0f}h total',
                 fontsize=12, weight='bold', pad=20, color=bar_color)

plt.suptitle('Daily Listening Patterns (Avg across All Days)\n'
             'Top 3 Playlists + All Others', fontsize=20, weight='bold', y=1.02)
plt.tight_layout()
plt.savefig('4_playlist_wheels_final.png', dpi=400, bbox_inches='tight', facecolor='white')
plt.show()